# Tutorial Básico de Gurobi para Programación Lineal

Este tutorial te enseñará los conceptos fundamentales para resolver problemas de programación lineal usando Gurobi en Python.

## Contenido:
1. Importar el módulo de Gurobi
2. Definir un problema ejemplo
3. Crear variables de decisión
4. Definir restricciones
5. Establecer la función objetivo
6. Resolver el problema
7. Extraer resultados y precios sombra


---

In [2]:
# Instalar la librería en caso de que no la tengas en tu dispositivo
!pip install gurobipy

## 1. Importación del Módulo Gurobi

Primero debemos importar la librería de Gurobi. Asegúrate de tener Gurobi instalado y con una licencia válida.

In [3]:
import gurobipy as gp
from gurobipy import GRB

# También importamos otra librerías útil para hacer DataFrames
import pandas as pd

## 2. Definición del Problema Ejemplo

Vamos a resolver un problema clásico de programación lineal:

**Problema de Producción:**

Una empresa produce dos productos: A y B. Queremos maximizar las ganancias.

- Producto A: ganancia de $3 por unidad
- Producto B: ganancia de $5 por unidad

**Restricciones:**
- Tiempo de máquina: 2 horas para A, 1 hora para B (máximo 100 horas disponibles)
- Material: 1 kg para A, 3 kg para B (máximo 80 kg disponibles)
- Demanda mínima: al menos 5 unidades de A
- Variables no negativas

**Formulación matemática:**

Maximizar: 3x₁ + 5x₂

Sujeto a:
- 2x₁ + x₂ ≤ 100  (tiempo de máquina)
- x₁ + 3x₂ ≤ 80   (material)
- x₁ ≥ 5          (demanda mínima)
- x₁, x₂ ≥ 0      (no negatividad)

## 3. Creación del Modelo

En Gurobi, todo problema se define dentro de un "modelo". Primero creamos el modelo:

In [4]:
# Crear el modelo
modelo = gp.Model("ProblemaProduccion")

Set parameter Username
Set parameter LicenseID to value 2817823
Academic license - for non-commercial use only - expires 2027-05-04


## 4. Definición de Variables de Decisión

Las variables de decisión representan las cantidades que queremos determinar. En nuestro caso, las cantidades a producir de cada producto.

Pese a ser positivas irrestrictas las variables, es buena práctica utilizar un número muy grande positivo como límite superior para mejorar la convergencia. Se denomina comúnmente `bigM` a este número muy grande. Para este ejemplo será 1 millón.

En este ejemplo las variables son continuas, por lo que su tipo será `GRB.CONTINUOUS`. Sin embargo, gurobi también permite definir otros tipos de variables, como binarias (`GRB.BINARY`) o enteras (`GRB.INTEGER`).

In [5]:
# Definir variables de decisión
# addVar(lb=lower_bound, ub=upper_bound, vtype=variable_type, name="nombre")

xa = modelo.addVar(lb=0, ub=1e6, vtype=GRB.CONTINUOUS, name="ProductoA") # Una variable puede ser Continuous, Binary o Integer
xb = modelo.addVar(lb=0, ub=1e6, vtype=GRB.CONTINUOUS, name="ProductoB")

# Actualizar el modelo para que reconozca las nuevas variables
modelo.update()

## 5. Definición de Restricciones

Las restricciones limitan el espacio de soluciones factibles. Se definen usando `addConstr()`.

In [6]:
# Definir restricciones

# Restricción 1: Tiempo de máquina (2x1 + x2 <= 100)
restriccion_tiempo = modelo.addConstr(2*xa + xb <= 100, name="TiempoMaquina")

# Restricción 2: Material (x1 + 3x2 <= 80)
restriccion_material = modelo.addConstr(xa + 3*xb <= 80, name="Material")

# Restricción 3: Demanda mínima (x1 >= 5)
restriccion_demanda = modelo.addConstr(xa >= 5, name="DemandaMinima")

Las restricciones de no negatividad $(x1, x2 >= 0)$ ya están incluidas en la definición de las variables con $lb=0$

## 6. Definición de la Función Objetivo

La función objetivo define qué queremos optimizar (maximizar o minimizar).

In [7]:
# Definir función objetivo
# Maximizar: 3x1 + 5x2
modelo.setObjective(3*xa + 5*xb, GRB.MAXIMIZE)

## 7. Resolución del Problema

Una vez definido completamente el modelo, usamos `optimize()` para resolverlo.

In [8]:
# Resolver el problema
modelo.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 3 rows, 2 columns and 5 nonzeros (Max)
Model fingerprint: 0x827fa6a8
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [3e+00, 5e+00]
  Bounds range     [1e+06, 1e+06]
  RHS range        [5e+00, 1e+02]

Presolve removed 1 rows and 0 columns
Presolve time: 0.01s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.6500000e+02   5.686200e+01   0.000000e+00      0s
       2    1.9200000e+02   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.920000000e+02


## 8. Extracción de Resultados

Si el problema tiene solución óptima, podemos extraer los valores de las variables y el valor objetivo.

In [9]:
# Extraer resultados si la solución es óptima
if modelo.status == GRB.OPTIMAL:
    print("=== RESULTADOS DE LA OPTIMIZACIÓN ===")
    print(f"Valor objetivo óptimo: ${modelo.objVal:.2f}")
    print()
    
    # Valores de las variables
    print("Valores óptimos de las variables:")
    print(f"Producto A (x1): {xa.x:.2f} unidades")
    print(f"Producto B (x2): {xb.x:.2f} unidades")
    print()
    
    # Verificar qué restricciones están activas (binding)
    print("Estado de las restricciones:")
    print(f"Tiempo de máquina: {2*xa.x + xb.x:.2f} / 100 horas")
    print(f"Material: {xa.x + 3*xb.x:.2f} / 80 kg")
    print(f"Demanda mínima: {xa.x:.2f} / 5 unidades mínimas")
else:
    print("No se pudo obtener una solución óptima")

=== RESULTADOS DE LA OPTIMIZACIÓN ===
Valor objetivo óptimo: $192.00

Valores óptimos de las variables:
Producto A (x1): 44.00 unidades
Producto B (x2): 12.00 unidades

Estado de las restricciones:
Tiempo de máquina: 100.00 / 100 horas
Material: 80.00 / 80 kg
Demanda mínima: 44.00 / 5 unidades mínimas


## 9. Extracción de Precios Sombra (Dual Values)

Los precios sombra nos indican cuánto aumentaría el valor objetivo si relajáramos cada restricción en una unidad.

In [10]:
# Extraer precios sombra (valores duales)
if modelo.status == GRB.OPTIMAL:
    print("=== PRECIOS SOMBRA (VALORES DUALES) ===")
    print()
    
    # Precios sombra de las restricciones
    print(f"Tiempo de máquina: ${restriccion_tiempo.pi:.2f} por hora adicional")
    print(f"Material: ${restriccion_material.pi:.2f} por kg adicional")
    print(f"Demanda mínima: ${restriccion_demanda.pi:.2f} por unidad adicional requerida")
    print()

=== PRECIOS SOMBRA (VALORES DUALES) ===

Tiempo de máquina: $0.80 por hora adicional
Material: $1.40 por kg adicional
Demanda mínima: $0.00 por unidad adicional requerida



# 10. Definición de MILP

Una empresa debe satisfacer una demanda de **100 ton/mes** de un producto. Existen tres sitios candidatos donde instalar una planta, y queremos **minimizar el costo total** (instalación + operación).

| Planta | Costo fijo $f_i$ | Costo variable $c_i$ | Capacidad $C_i$ |
|:------:|:----------------:|:--------------------:|:---------------:|
| P1 | 500 | 8 | 60 |
| P2 | 300 | 12 | 50 |
| P3 | 700 | 5 | 80 |

**Restricciones**:

- **Demanda**: la producción total de las plantas instaladas debe ser exactamente 100 ton/mes.
- **Capacidad**: cada planta produce a lo más su capacidad, y solo puede producir si fue instalada.
- **Decisión de instalación**: para cada sitio se decide si se instala o no la planta (variable binaria).
- **Variables no negativas**.


Sea $x_i \geq 0$ la producción de la planta $i$ e $y_i \in \{0,1\}$ la decisión de instalarla:

$$
\begin{aligned}
\min_{\substack{x_1,\,x_2,\,x_3 \\ y_1,\,y_2,\,y_3}} \quad
& 500\,y_1 + 300\,y_2 + 700\,y_3 + 8\,x_1 + 12\,x_2 + 5\,x_3 \\[4pt]
\text{s.a.} \quad
& x_1 + x_2 + x_3 = 100 \\
& x_1 \leq 60\,y_1 \\
& x_2 \leq 50\,y_2 \\
& x_3 \leq 80\,y_3 \\
& x_1,\, x_2,\, x_3 \geq 0 \\
& x_1,\, x_2,\, x_3 \in \mathbb{R} \\
& y_1,\, y_2,\, y_3 \in \{0,1\}
\end{aligned}
$$


# 11. Definición problema MILP en Gurobipy

In [11]:
model_milp = gp.Model("ProblemaProduccionMILP")
# Definir variables de decisión
x1 = model_milp.addVar(lb=0, ub=1e6, vtype=GRB.CONTINUOUS, name="ProductoA") 
x2 = model_milp.addVar(lb=0, ub=1e6, vtype=GRB.CONTINUOUS, name="ProductoB")
x3 = model_milp.addVar(lb=0, ub=1e6, vtype=GRB.CONTINUOUS, name="ProductoC")
y1 = model_milp.addVar(vtype=GRB.BINARY, name="Indicador")  
y2 = model_milp.addVar(vtype=GRB.BINARY, name="Indicador2")
y3 = model_milp.addVar(vtype=GRB.BINARY, name="Indicador3")

## agregar restricciones
restriccionProd = model_milp.addConstr(x1 + x2 + x3 == 100, name="ProdTotal")
# capacidades productivas
restriccionCap1 = model_milp.addConstr(x1 <= 60 * y1, name="CapacidadProdA")
restriccionCap2 = model_milp.addConstr(x2 <= 50 * y2, name="CapacidadProdB")
restriccionCap3 = model_milp.addConstr(x3 <= 80 * y3, name="CapacidadProdC")
model_milp.update()
# Definir función objetivo
model_milp.setObjective(500 * y1 + 300 * y2 + 700 * y3 + 8 * x1 + 12 * x2 + 5 * x3, GRB.MINIMIZE)
# Resolver el problema
model_milp.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 4 rows, 6 columns and 9 nonzeros (Min)
Model fingerprint: 0xb3078906
Model has 6 linear objective coefficients
Variable types: 3 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+01]
  Objective range  [5e+00, 7e+02]
  Bounds range     [1e+00, 1e+06]
  RHS range        [1e+02, 1e+02]

Presolve time: 0.01s
Presolved: 4 rows, 6 columns, 9 nonzeros
Variable types: 3 continuous, 3 integer (3 binary)
Found heuristic solution: objective 2060.0000000

Root relaxation: objective 1.426667e+03, 3 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 1426.66667    0    1 2060.00000 1426.66667  30.7%   

# 12. Extracción del resultado

In [21]:
# Extraer resultados si la solución es óptima
if modelo.status == GRB.OPTIMAL:
    print("=== RESULTADOS DE LA OPTIMIZACIÓN ===")
    print(f"Valor objetivo óptimo: ${model_milp.objVal:.2f}")
    print()
    
    # Valores de las variables
    print("Valores óptimos de las variables:")
    print(f"Producto A (x1): {x1.x:.2f} unidades")
    print(f"Producto B (x2): {x2.x:.2f} unidades") 
    print(f"Producto C (x3): {x3.x:.2f} unidades")
    print('====== Construcción planta ======')
    print(f"Planta A: {y1.x}")
    print(f"Planta B: {y2.x}")
    print(f"Planta C: {y3.x}")
    
    print()
    
    # Verificar qué restricciones están activas (binding)
    print("Estado de las restricciones:")
    print(f"Restricción de producción total: {x1.x + x2.x + x3.x:.2f} / 100 unidades")
    print(f"Capacidad de producción A: {x1.x:.2f} / 60 * {y1.x:.2f} unidades")
    print(f"Capacidad de producción B: {x2.x:.2f} / 50 * {y2.x:.2f} unidades")
    print(f"Capacidad de producción C: {x3.x:.2f} / 80 * {y3.x:.2f} unidades")
else:
    print("No se pudo obtener una solución óptima")

=== RESULTADOS DE LA OPTIMIZACIÓN ===
Valor objetivo óptimo: $1640.00

Valores óptimos de las variables:
Producto A (x1): 0.00 unidades
Producto B (x2): 20.00 unidades
Producto C (x3): 80.00 unidades
====== Construcción planta ======
Planta A: 0.0
Planta B: 1.0
Planta C: 1.0

Estado de las restricciones:
Restricción de producción total: 100.00 / 100 unidades
Capacidad de producción A: 0.00 / 60 * 0.00 unidades
Capacidad de producción B: 20.00 / 50 * 1.00 unidades
Capacidad de producción C: 80.00 / 80 * 1.00 unidades


# 13. Problemas de optimización en formato matricial

Una planta debe formular **100 kg** de un alimento a partir de cuatro materias primas: maíz ($M_1$), harina de soya ($M_2$), afrecho de trigo ($M_3$) y aceite vegetal ($M_4$). Queremos **minimizar el costo** de la mezcla.

Composición de cada materia prima (kg de componente por kg de materia prima) y costo:

|              | $M_1$ | $M_2$ | $M_3$ | $M_4$ |
|:-------------|:-----:|:-----:|:-----:|:-----:|
| Proteína     | 0.09  | 0.45  | 0.15  | 0.00  |
| Grasa        | 0.04  | 0.02  | 0.04  | 1.00  |
| Fibra        | 0.02  | 0.06  | 0.10  | 0.00  |
| **Costo** [USD/kg] | 0.30 | 0.90 | 0.35 | 1.50 |

**Restricciones**:

- **Balance de masa**: la suma de las cuatro materias primas debe ser exactamente 100 kg.
- **Especificación nutricional**: la mezcla debe contener al menos 20 kg de proteína y al menos 5 kg de grasa.
- **Límite de fibra**: la mezcla debe contener a lo más 5 kg de fibra.
- **Disponibilidad**: se dispone de a lo más 10 kg de aceite vegetal.
- **Variables no negativas**.


Sea $\mathbf{x} \in \mathbb{R}^4$ el vector con los kilos de cada materia prima:

$$
\begin{aligned}
\min_{\mathbf{x}} \quad & \mathbf{c}\cdot\mathbf{x} \\[4pt]
\text{s.a.} \quad
& \mathbf{A}\mathbf{x} \leq \mathbf{b} \\
& \mathbf{1}\cdot\mathbf{x} = 100 \\
& \mathbf{0} \leq \mathbf{x} \leq \mathbf{x}^{UB} \\
& \mathbf{x} \in \mathbb{R}^{4}
\end{aligned}
$$

donde

$$
\mathbf{c}=\begin{bmatrix}0.30\\0.90\\0.35\\1.50\end{bmatrix},\quad
\mathbf{A}=\begin{bmatrix}
-0.09 & -0.45 & -0.15 &  0.00\\
-0.04 & -0.02 & -0.04 & -1.00\\
 0.02 &  0.06 &  0.10 &  0.00
\end{bmatrix},\quad
\mathbf{b}=\begin{bmatrix}-20\\-5\\5\end{bmatrix},\quad
\mathbf{x}^{UB}=\begin{bmatrix}100\\100\\100\\10\end{bmatrix}
$$


In [16]:
import numpy as np
model_matrix = gp.Model('matrixProblem')
# Definir restricciones
A = np.array([
    [-0.09, -0.45, -0.15, 0],
    [-0.04, -0.02, -0.04, -1],
    [0.02, 0.06, 0.1, 0]
])
c = np.array([0.3, 0.9, 0.35, 1.5])
b = np.array([-20, -5, 5])
x_ub = np.array([100, 100, 100, 10])
n = A.shape[1]

# Variables
x = model_matrix.addMVar(shape=n, lb=0, ub=x_ub, name="x")

# Restricciones
matCons = model_matrix.addConstr(A @ x <= b, name="Matconstraints")
sumCons = model_matrix.addConstr(np.ones(n) @ x == 100, name="sum_constraint")

# Objetivo
model_matrix.setObjective(c @ x, GRB.MINIMIZE)
model_matrix.optimize()



Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 4 rows, 4 columns and 14 nonzeros (Min)
Model fingerprint: 0x84aa99d4
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-02, 1e+00]
  Objective range  [3e-01, 2e+00]
  Bounds range     [1e+01, 1e+02]
  RHS range        [5e+00, 1e+02]

Presolve removed 1 rows and 0 columns
Presolve time: 0.01s
Presolved: 3 rows, 4 columns, 11 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.4166667e+01   1.143373e+01   0.000000e+00      0s
       3    4.9272518e+01   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.01 seconds (0.00 work units)
Optimal objective  4.927251824e+01


In [19]:
# Etiquetas
materias = ["Maíz (M1)", "Harina de soya (M2)", "Afrecho de trigo (M3)", "Aceite vegetal (M4)"]
componentes = ["Proteína", "Grasa", "Fibra"]

# Matriz de composición física (sin el cambio de signo) y especificaciones
A_c = np.array([
    [0.09, 0.45, 0.15, 0.00],
    [0.04, 0.02, 0.04, 1.00],
    [0.02, 0.06, 0.10, 0.00]
])
specs = [(">=", 20.0), (">=", 5.0), ("<=", 5.0)]

# Extraer resultados si la solución es óptima
if model_matrix.Status == GRB.OPTIMAL:
    print("=== RESULTADOS DE LA OPTIMIZACIÓN ===")
    print(f"Costo mínimo de la mezcla: ${model_matrix.ObjVal:.2f}")
    print()

    # Valores de las variables
    print("Valores óptimos de las variables:")
    for nombre, valor, ub in zip(materias, x.X, x_ub):
        print(f"  {nombre:22s}: {valor:6.2f} kg   (máx: {ub:5.1f} kg)")
    print()

    # Verificar qué restricciones están activas
    print("Estado de las restricciones:")
    print(f"  Balance de masa : {x.X.sum():6.2f} / 100.00 kg   [ACTIVA]")
    kg_comp = A_c @ x.X
    for k, nombre in enumerate(componentes):
        signo, lim = specs[k]
        holgura = abs(kg_comp[k] - lim)
        estado = "ACTIVA" if holgura < 1e-6 else f"holgura = {holgura:.2f} kg"
        print(f"  {nombre:16s}: {kg_comp[k]:6.2f} {signo} {lim:6.2f} kg   [{estado}]")
    print()

    # Precios sombra (solo válidos para LP)
    print("Precios sombra [USD/kg de componente]:")
    for nombre, pi in zip(componentes, matCons.Pi):
        print(f"  {nombre:16s}: {pi:7.4f}")
    print(f"  {'Balance de masa':16s}: {sumCons.Pi:7.4f}")
else:
    print("No se pudo obtener una solución óptima")

=== RESULTADOS DE LA OPTIMIZACIÓN ===
Costo mínimo de la mezcla: $49.27

Valores óptimos de las variables:
  Maíz (M1)             :  47.06 kg   (máx: 100.0 kg)
  Harina de soya (M2)   :  26.88 kg   (máx: 100.0 kg)
  Afrecho de trigo (M3) :  24.46 kg   (máx: 100.0 kg)
  Aceite vegetal (M4)   :   1.60 kg   (máx:  10.0 kg)

Estado de las restricciones:
  Balance de masa : 100.00 / 100.00 kg   [ACTIVA]
  Proteína        :  20.00 >=  20.00 kg   [ACTIVA]
  Grasa           :   5.00 >=   5.00 kg   [ACTIVA]
  Fibra           :   5.00 <=   5.00 kg   [ACTIVA]

Precios sombra [USD/kg de componente]:
  Proteína        : -1.8276
  Grasa           : -1.4058
  Fibra           : -0.7457
  Balance de masa :  0.0942
